# Gemma Scope 2 Tutorial

This is a tutorial on how to use [Gemma Scope 2](https://huggingface.co/google/gemma-scope-2), Google DeepMind's suite of Sparse Autoencoders (SAEs) and Transcoders (TCs) on every layer of every model in the Gemma V3 family, as well as several multi-layer models.

Sparse Autoencoders are interpretability tools that act like a "microscope" on language model activations. They let us zoom in on dense, compressed activations, and expand them to a larger but sparser and seemingly more interpretable form, which can be a very useful tool when doing interpretability research! Transcoders and multi-layer models expand on this by helping us find not just individual concepts but circuits connecting concepts together, unlocking understanding of more complex behaviours.

**Learn more:**
* If you want to learn about Gemma Scope without writing any code, check out [this interactive demo](https://neuronpedia.org/gemma-scope-2) courtesy of [Neuronpedia](https://neuronpedia.org).
* For an overview of Gemma Scope check out [the blog post](https://deepmind.google/blog/gemma-scope-2-helping-the-ai-safety-community-deepen-understanding-of-complex-language-model-behavior/).
* See [the technical report](https://storage.googleapis.com/deepmind-media/DeepMind.com/Blog/gemma-scope-2-helping-the-ai-safety-community-deepen-understanding-of-complex-language-model-behavior/Gemma_Scope_2_Technical_Paper.pdf) for the technical details

For illustrative purposes, we begin with a lightweight tutorial that uses as few libraries as possible to outline how Gemma Scope works, and what Sparse Autoencoders are doing. This is deliberately a fairly minimalist tutorial, designed to make clear what is actually going on, but does not model research best practices.

For any serious research with Gemma Scope, **we recommend using the [SAELens](https://decoderesearch.github.io/SAELens/) and [TransformerLens](https://transformerlensorg.github.io/TransformerLens/) libraries**, see [this tutorial](https://colab.research.google.com/github/decoderesearch/SAELens/blob/main/tutorials/tutorial_2_0.ipynb) on how to use [SAELens](https://decoderesearch.github.io/SAELens/) in practice.


## Loading the Model

First, let's load the model:

For simplicity we do this straight from [HuggingFace transformers](https://huggingface.co/docs/transformers/en/index), rather than using an interpretability focused library like [TransformerLens](https://transformerlensorg.github.io/TransformerLens/) or [nnsight](https://nnsight.net/).

In [1]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from huggingface_hub import hf_hub_download, notebook_login
import numpy as np
import einops
import textwrap
from typing import Literal
import plotly.express as px
from functools import partial
import dataclasses
from IPython.display import display, HTML
import gc
import pandas as pd
from safetensors.torch import load_file
import torch
import torch.nn as nn

In [2]:
!nvidia-smi

Mon Mar  9 00:57:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  NVIDIA H100 NVL                Off |   00000000:E1:00.0 Off |                    0 |
| N/A   31C    P0             60W /  400W |       0MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+------------------------+----------------------+
                                                                                         
+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                     

We load Gemma 3 1B, the second smallest model that Gemma Scope 2 works for (you can also try Gemma 3 270m, but in a Colab the 1B-size model should work fine).

We load the base model, not the chat model, since that's where our SAEs are trained. Though the SAEs seem to transfer OK to these models. First, you'll need to authenticate with huggingface in order to download the model weights (you only need a **READ** token; no other permissions).

In [3]:
notebook_login()

In [4]:
torch.set_grad_enabled(False) # avoid blowing up mem

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-pt",
    device_map='auto',
)
tokenizer =  AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")

Now we've loaded the model, let's try running it! We give it a prompt (which we'll come back to later) and print the generated output:

In [30]:
# The input text
prompt_physics = "The law of conservation of energy states that energy cannot be created or destroyed, only transformed."
# prompt_physics = "Please say a sentence in french, and then a second sentence in german (can be anything)"

# Use the tokenizer to convert it to tokens
# Note that this implicitly adds a special "Beginning of Sequence" or <bos> token to the start
inputs_physics = tokenizer.encode(prompt_physics, return_tensors="pt", add_special_tokens=True).to("cuda")
print(inputs_physics)

# Pass it in to the model and generate text
outputs = model.generate(input_ids=inputs_physics, max_new_tokens=50)
output_str = tokenizer.decode(outputs[0])
print()
print(textwrap.fill(output_str))

tensor([[     2,   9366,   1879,    496,  13315,    528,  38986, 236764,    532,
           1299,    496,   1855,  13315,    528,  47086,    568,   4881,    577,
           4658, 236768]], device='cuda:0')

<bos>Please say a sentence in french, and then a second sentence in
german (can be anything) and tell me your translation.  Hi, I hope
you're well. I think the translation of the sentences I gave you is...
it's a question of point of view. I'll ask a friend to help me. Thanks


This was the pretrained (PT) model, so it doesn't respond like a chatbot - it just continues based on its priors for what is likely to follow the initial prompt, given the dataset it was trained on.

We'll also be using the instruction-tuned (IT) model, which behaves more like a standard chatbot.Let's also load that in and see how it works. Note that we have to carefully format the input so that it's in the correct form for our IT model:

In [31]:
model_it = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map='auto',
)

In [33]:
def format_prompt(user_prompt: str) -> str:
  return f"""<start_of_turn>user
{user_prompt}<end_of_turn>
<start_of_turn>model
"""

user_prompt = "What is your name?"
user_prompt = "please output a sentence in french, then a second sentence in german (can be anything)"
it_inputs = tokenizer.encode(format_prompt(user_prompt), return_tensors="pt", add_special_tokens=True).to("cuda")

outputs = model_it.generate(input_ids=it_inputs, max_new_tokens=60)
print(tokenizer.decode(outputs[0]))

<bos><start_of_turn>user
please output a sentence in french, then a second sentence in german (can be anything)<end_of_turn>
<start_of_turn>model
Okay, here's a sentence in French and a second sentence in German:

**French:**  "Le chat dort paisiblement sur le tapis rouge."

**German:** "Der Himmel ist blau und die Sonne scheint."
<end_of_turn>


## Sparse Autoencoders

OK, so we have got our Gemma model loaded, and we can sample from it to get sensible stuff. Now, let's load one of our SAEs.

GemmaScope actually contains over four hundred SAEs, but for now we'll just load one on the residual stream at the end of layer 20 (of 26, note that layers start at 0 so this is the 21st layer. This is a fairly late layer, so the model should have time to find more abstract concepts!).

Note, we're loading from the `resid_post` directory here. We can also load from `resid_post_all` to get SAEs trained on every single layer (not just a subset of 4 layers), but a smaller range of widths and L0 values.

<details><summary>What is the residual stream?</summary>

Transformers have skip connections, which means that the output of each block is the output of each sublayer *plus* the input to the block. This means that each sublayer (attention or MLP) actually only has a fairly small effect on the output of the block, since most of it comes from all the earlier layers. We call the output of a block (including skip connections) the **residual stream**.

Everything communicated from earlier layers to later layers must go via the residual stream, so it acts as a "bottleneck" in the transformer, essentially capturing everything the model has "thought" so far. This means it is often a natural thing to study, since it will contain everything important going on in the model.
</details>


In [8]:
LAYER = 22  # options are {7, 13, 17, 22}
WIDTH = "65k"   # options are {16k, 65k, 262k, 1m}
L0 = "medium"  # options are {small, medium, big}

path_to_params = hf_hub_download(
    repo_id="google/gemma-scope-2-1b-pt",
    filename=f"resid_post/layer_{LAYER}_width_{WIDTH}_l0_{L0}/params.safetensors",
)

params = load_file(path_to_params)

Our SAEs are **JumpReLU** SAEs, meaning they are a standard 2-layer neural network with a JumpReLU activation function (a ReLU with a discontinuous jump).

<!-- The mapping from input to hidden activations is defined by the weight and bias parameters `W_enc` and `b_enc`, and the mapping from hidden activations back to reconstructed input is defined by `W_dec` and `b_dec`. The `threshold` parameter determines the size of the discontinuity for JumpReLU. -->

### Implementing the SAE


We now define the forward pass of the SAE for pedagogical purposes (in practice, we recommend using the implementation in SAELens).

We have 5 important parameters below:

- `w_enc`, the encoder matrix (which maps from inputs to pre-activation latent values)
- `b_enc`, the bias added onto these pre-activation latent values
- `threshold`, which determines how we apply our JumpReLU activation function
- `w_dec`, the decoder matrix (which maps from post-ReLU latent values to reconstructed activations)
- `b_dec`, the bias which is added to the final reconstruction

You can ignore `affine_skip_connection` for now; we'll come back to it in the "transcoders" section.

In [9]:
class JumpReLUSAE(nn.Module):
  def __init__(self, d_in, d_sae, affine_skip_connection=False):
    # Note that we initialise these to zeros because we're loading in pre-trained weights.
    # If you want to train your own SAEs then we recommend using blah
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
    self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
    self.threshold = nn.Parameter(torch.zeros(d_sae))
    self.b_enc = nn.Parameter(torch.zeros(d_sae))
    self.b_dec = nn.Parameter(torch.zeros(d_in))
    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    pre_acts = input_acts @ self.w_enc + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return acts @ self.w_dec + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + x @ self.affine_skip_connection
    return recon

In [10]:
d_model, d_sae = params["w_enc"].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()

JumpReLUSAE()

### Running the SAE on model activations


Let's first get out some activations from the model at the SAE target site. We'll demonstrate how to do this 'manually' first, by using Pytorch hooks. Note that this is not particularly good practice, and it's probably more practical to use a library like TransformerLens to handle hooking the SAE into a model forward pass. But for illustrative purposes, it's useful to see how it's done.

We can gather activations at a site by registering a hook. To keep this local, we can wrap this in a function that registers a hook, runs the model, saving the intermediate activation, then removes the hook. (This is basically what TransformerLens is doing under the hood)

In [11]:
def gather_acts_hook(mod, inputs, outputs, cache: dict, key: str, use_input: bool):
  """Generic hook function whic stores activations (either input or output of a particular PyTorch module)."""
  acts = inputs[0].squeeze(0) if use_input else outputs[0]  # inputs usually have a batch dim
  cache[key] = acts
  return outputs


def gather_residual_activations(model, target_layer, inputs):

  cache = {}

  # Add a hook function to store the output of this layer of the model
  handle = model.model.layers[target_layer].register_forward_hook(
      partial(gather_acts_hook, cache=cache, key="resid_post", use_input=False)
  )

  # Forward pass inside a try/except/finally block (useful just in case our hook breaks
  # and we can't remove it!)
  try:
    _ = model.forward(inputs)
  finally:
    handle.remove()

  return cache["resid_post"]

In [12]:
target_act = gather_residual_activations(model, LAYER, inputs_physics)

Now, we can run our SAE on the saved activations.

In [13]:
sae_acts = sae.encode(target_act.to(torch.float32))
recon = sae.decode(sae_acts)

Let's just double check that the model looks sensible by checking that we explain a decent chunk of the variance:

In [14]:
reconstruction_mse = torch.mean((recon[:, 1:] - target_act[:, 1:].float()) ** 2)
target_variance = target_act[:, 1:].float().var()

fvu = reconstruction_mse / target_variance
print(f"Fraction of variance unexplained: {fvu:.2%}")

Fraction of variance unexplained: 2.27%


This looks pretty good!

This SAE is supposed to have an L0 of ~60 (size "medium"), so let's check that too:

In [15]:
l0_per_token = (sae_acts > 1).sum(-1)[0]
print(l0_per_token.tolist())

print(f"Average L0: {l0_per_token[1:].float().mean():.2f}")

[87, 14, 55, 88, 50, 85, 52, 68, 98, 86, 88, 115, 72, 61, 50, 58, 90, 70, 58]
Average L0: 69.89


It's always worth checking this sort of thing when you do this by hand to check that you haven't got the wrong site, or are missing a scaling factor or something like this. But here, our results all look like they are supposed to .

Note that there's a bit of a gotcha here; our SAEs are *NOT* trained on the BOS token, because we found that this tended to be a large outlier and to mess up training. So they tend to give nonsense when we apply to them to it, and we need to be careful not to do this accidentally! We can see this above : the BOS token is a total outlier in terms of L0!

Another way we can evaluate our SAE is by looking at the **delta loss**, i.e. how much the model's prediction loss increases when we patch in the SAE's output. To do this we'll set up a new hook function:

In [16]:
def fwd_pass_with_sae_intervention(model, sae, target_layer, inputs):
  # Forward pass to get logits & hidden activations
  model_output_clean = model.forward(inputs, output_hidden_states=True)
  logits_clean = model_output_clean.logits[0]  # (seq, d_vocab)
  input_acts = model_output_clean.hidden_states[target_layer + 1][0]  # (seq, d_model)

  # Get the SAE reconstruction
  recon = sae.forward(input_acts.to(torch.float32))

  def intervene_on_target_act_hook(mod, inputs, outputs):
    outputs[0][0, 1:] = recon[1:]
    return outputs

  handle = model.model.layers[target_layer].register_forward_hook(intervene_on_target_act_hook)
  try:
    model_output = model.forward(inputs)
  finally:
    handle.remove()

  # Get logits from this corrupted forward pass
  logits = model_output.logits[0]

  return logits_clean, logits


def cross_entropy_loss(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
  """Measures avg cross entropy loss."""
  logprobs = logits[:-1].log_softmax(dim=-1)
  tokens = tokens[1:]
  correct_logprobs = logprobs[torch.arange(len(tokens)), tokens]
  return -correct_logprobs

In [17]:
logits_clean, logits_sae = fwd_pass_with_sae_intervention(model, sae, LAYER, inputs_physics)
loss_clean = cross_entropy_loss(logits_clean, inputs_physics[0])
loss_sae = cross_entropy_loss(logits_sae, inputs_physics[0])

print(f"Loss (clean): {loss_clean.mean():.4f}")
print(f"Loss (corrupted): {loss_sae.mean():.4f}")
print(f"Delta loss: {loss_sae.mean() - loss_clean.mean():.4f}")

Loss (clean): 1.3977
Loss (corrupted): 1.5646
Delta loss: 0.1669


Let's look at the highest activating features on this input text, on each token position:

In [22]:
model.model.layers[0]

Gemma3DecoderLayer(
  (self_attn): Gemma3Attention(
    (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
    (k_proj): Linear(in_features=1152, out_features=256, bias=False)
    (v_proj): Linear(in_features=1152, out_features=256, bias=False)
    (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
    (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
    (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
  )
  (mlp): Gemma3MLP(
    (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
    (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
    (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
    (act_fn): GELUTanh()
  )
  (input_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (post_attention_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (pre_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
  (post_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
)

In [20]:
sae_acts.shape

torch.Size([1, 19, 65536])

In [18]:
top_activations, top_features = sae_acts.max(-1)

top_features

tensor([[ 1904,  4489,   990, 62854,   195,   773,  1665,  1643,  6524, 50465,
          2622,  1138,   643,  1138,   119,  3419,  1138,  7188,  1604]],
       device='cuda:0')

Note that a lot of these indices are quite small, relative to the number of features in our SAE (262 thousand in this case). This is because our SAE was trained with [**Matryoshka loss**](https://www.lesswrong.com/posts/zbebxYCqsryPALh8C/matryoshka-sparse-autoencoders), which imposes a feature hierarchy: the smaller-indexed features are incentivised to be good at reconstructing the input even when all other features are switched off. This helps avoid problems like **feature absorption**.

<!-- Now, let's take a particular word in our prompt from earlier, and see which feature causes it to fire strongest: -->

<!-- It seems like **1473** is a pretty common top-activating feature in the sequence above, so let's inspect this feature. First, we'll see exactly which tokens in our prompt it fires on: -->

<!-- Let's inspect this feature. First, we'll see exactly which tokens in our prompt it fires on: -->

Let's find the feature which activates the strongest when averaged over all tokens in the sequence:

<!--
gravity_idx = tokenizer.tokenize(prompt_newton, add_special_tokens=True).index("▁universe")
print(f"Token idx: {gravity_idx}")

feature_idx = top_features.squeeze().tolist()[gravity_idx]
print(f"Top activating feature: {feature_idx}")

feature_idx = top_features.squeeze().mean(0).argmax()
print(f"Top activating feature: {feature_idx}")
-->

In [24]:
top_acts, top_latents = sae_acts.squeeze().mean(0).topk(5)

for act, idx in zip(top_acts, top_latents):
  print(f"{act:>6.1f} | {idx}")

 889.1 | 6524
 820.0 | 1854
 669.5 | 21489
 657.2 | 1904
 585.8 | 4181


Latent 5698 seems to fire strongest, so let's inspect it.

<!-- Latent 5698 seems to fire strongest, but since we're training Matryoshka SAEs (where higher-indexed features are generally lower-frequency and represent more specific concepts), let's inspect the largest feature in the top 5: feature 96190. -->

In [25]:
feature_idx = 6524

str_toks = tokenizer.tokenize(prompt_physics, add_special_tokens=True)
activations = sae_acts[0, :, feature_idx].tolist()

def html_activations(str_toks: list[str], activations: list[float]):
  return "".join(
      f'<span style="background-color: rgba(255,0,0,{v}); padding: 4px 0px;">{t}</span>'
      for t, v in zip(str_toks, np.array(activations) / (1e-6 + np.max(activations)), strict=True)
  )

display(HTML(html_activations(str_toks, activations)))

One guess we might have is that this latent fires on concepts related to science or scientific laws. Let's test this out with a few examples:

In [26]:
for prompt in [
    "Gemma Scope 2 is a model release from Google DeepMind",
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit",
    "Gravity describes how massive objects attract one another",
    "A charge accelerating through an electric field experiences a force",
    "Chemical fuel stores energy in molecular bonds, which is released"
]:
  inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")
  _target_acts = gather_residual_activations(model, LAYER, inputs)

  _sae_acts = sae.encode(_target_acts.to(torch.float32))

  str_toks = tokenizer.tokenize(prompt, add_special_tokens=True)
  display(HTML(html_activations(str_toks, _sae_acts[0, :, feature_idx].tolist())))
  print()

Okay, so it doesn't fire on the gravity sentence, but it does fire on both the other physics-related sentences as soon as they start talking about forces, energies or fields. This gives us a more specific idea of the concepts this latent might represent.

<!-- This theory seems reasonable: it fires on both the sentences related to physical forces (note that it doesn't seem to just be a "gravity" latent given how it fires on the second of these two sentences). -->

<!-- Okay, so it seems like this might be the case, although the activation is more consistent when describing gravity than on other forces. -->

We can investigate this further by looking at the latent's unembedding, in other words **what words get predicted strongest when this latent fires.** From the results below, we might guess this latent represents a more specific concept: entropy and thermodynamics.

In [27]:
w_u = model.lm_head.weight  # shape (d_vocab, d_model)
w_u_eff = w_u * model.model.norm.weight

decoder_vector = sae.w_dec[feature_idx]  # shape (d_model,)

top_activations, top_tokens = torch.topk(w_u_eff @ decoder_vector, k=10)

for act, tok in zip(top_activations, top_tokens):
  print(f"{act:.4f} | {tokenizer.decode(tok)}")

1.1846 |  entropy
1.1499 |  Entropy
1.0740 |  enthalpy
1.0628 |  calorimetric
1.0627 |  entrop
1.0452 |  Gibb
1.0451 | Entropy
1.0359 | 焓
1.0186 |  Gibbs
0.9941 |  enthal


<!-- This definitely increases credence in our theory that this feature specifically relates to gravity! -->

Lastly, we can try **steering with this feature**. This means intervening in the residual stream of the model to add some multiple of this feature's decoder vector, so that we can change the behaviour of the model during generation.

You should see that when we steer the model on this "physical force feature", it starts talking more about enetry (specifically entropy or thermodynamics). Note that steering can often be fragile; it's difficult to choose the intervention layer and steering coefficient in a way that gives the expected behavioural change without also breaking the model's coherence. If you're curious, you can try increasing the `coeff` parameter below and seeing what happens!

In [28]:
def generate_with_steering(model, sae, inputs, target_layer, feature_idx: int, coeff: float):

  def steering_hook(mod, inputs, outputs):
    output = outputs[0]
    # We have to be careful about KV caching! This logic handles different cases depending on
    # whether this is the first forward pass or a cached pass.
    if output.shape[1] == 1:
      avg_norm = torch.norm(output, dim=-1)
      output += coeff * avg_norm * sae.w_dec[feature_idx]
    else:
      # avg_norm = torch.norm(output[0, 1:], dim=-1, keepdim=True)
      # output[0, 1:] += coeff * avg_norm * sae.w_dec[feature_idx]
      avg_norm = torch.norm(output[0, -1:], dim=-1, keepdim=True)
      output[0, -1:] += coeff * avg_norm * sae.w_dec[feature_idx]

    return outputs

  handle = model.model.layers[target_layer].register_forward_hook(steering_hook)
  try:
    outputs = model.generate(input_ids=inputs, max_new_tokens=80, do_sample=False)
    output_str = tokenizer.decode(outputs[0])
  finally:
    handle.remove()

  return output_str.split("<start_of_turn>model")[1].strip()


user_prompt = "Tell me a fun fact."
inputs = tokenizer.encode(format_prompt(user_prompt), return_tensors="pt", add_special_tokens=True).to("cuda")

print(user_prompt)
print("======================= NO STEERING =======================")
output_str = generate_with_steering(
    model=model_it,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.0,
)
print(textwrap.fill(output_str))
print("======================= STEERING =======================")
output_str_steered = generate_with_steering(
    model=model_it,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.14,
)
print(textwrap.fill(output_str_steered))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tell me a fun fact.
======================= NO STEERING =======================
Okay, here’s a fun fact for you:  **Octopuses have three hearts!**
Two pump blood through the gills, and one pumps it to the rest of the
body.   Pretty cool, right? 🐙❤️‍🔥  Do you want to hear another fun
fact?<end_of_turn>
======================= STEERING =======================
Okay, here’s a fun fact:  **There’s a species of jellyfish called
*Psychrolestes lunare* that’s named after the moon’s surface,
specifically because it’s named after the moon’s thermal gradient.**
That’s a mouthful, right?   Basically, it’s a cold-resistant jellyfish
that lives in the depths of the ocean,


Note that steering is expected to be pretty brittle with smaller models. Generally, larger models (up to a certain point) can better express more complex concepts and are easier to steer without breaking coherence.

As an exercise, try finding more latents to steer with. Can you come up with any other interesting prompts and latents?

# Other models

We finish with a speedrun of other models in the GemmaScope 2 release. These include:

- Single-layer sparse autoencoders trained on **attention output** and **MLP output**
- [Weakly-causal Crosscoders](https://transformer-circuits.pub/2024/crosscoders/index.html#cross-layer-features) trained on 4 concatenated layers of the residual stream
- [Cross-layer transcoders](https://transformer-circuits.pub/2025/attribution-graphs/methods.html), trained to reconstruct all concatenated MLP outputs from all residual stream activations

## MLP-output and attention-output SAEs

In [ ]:
def gather_mlp_out_activations(model, target_layer, inputs):
  act_cache = {}
  handle = model.model.layers[target_layer].post_feedforward_layernorm.register_forward_hook(
      partial(gather_acts_hook, key="acts", cache=act_cache, use_input=False)
  )
  try:
    _ = model.forward(inputs)
  finally:
    handle.remove()
  return act_cache.pop("acts")


def gather_attn_out_activations(model, target_layer, inputs):
  act_cache = {}
  handle = model.model.layers[target_layer].self_attn.o_proj.register_forward_hook(
      partial(gather_acts_hook, key="acts", cache=act_cache, use_input=True)
  )
  try:
    _ = model.forward(inputs)
  finally:
    handle.remove()
  return act_cache.pop("acts")

First, the MLP-output SAE:

In [ ]:
def load_sae(category: str, layer: int, width: int, l0: str, affine: bool = False) -> JumpReLUSAE:
  affine_str = "_affine" if affine else ""
  path_to_params = hf_hub_download(
      repo_id="google/gemma-scope-2-1b-pt",
      filename=f"{category}/layer_{layer}_width_{width}_l0_{l0}{affine_str}/params.safetensors",
  )
  params = load_file(path_to_params)
  d_model, d_sae = params["w_enc"].shape
  sae = JumpReLUSAE(d_model, d_sae)
  sae.load_state_dict(params)
  return sae.cuda()

In [ ]:
mlp_sae = load_sae(
    category="mlp_out",
    layer=17,
    width="16k",
    l0="medium"
)

prompt = "The quick brown fox jumped over the lazy dog"
inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

sae_input = gather_mlp_out_activations(model, LAYER, inputs)

sae_acts = mlp_sae.encode(sae_input.to(torch.float32))
recon = mlp_sae.decode(sae_acts)

mse = torch.mean((recon[1:] - sae_input[1:].float())**2)
var = sae_input[1:].float().var()
fvu = mse / var
l0 = (sae_acts[1:] > 0).float().sum(-1).mean()

print(f"L0: {l0:.2f}")
print(f"Fraction of variance unexplained: {mse / var:.2%}")

mlp_out/layer_17_width_16k_l0_medium/par(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

L0: 121.56
Fraction of variance unexplained: 17.13%


Next, we'll look at the **attention output SAE**, which reconstructs the pre-layernorm values of the attention layer. Note that this means if you want to do any kind of circuit analysis, you should probably be freezing layernorm (and remember that these values are summed over the head dimension).

In [ ]:
attn_sae = load_sae(
    category="attn_out",
    layer=17,
    width="16k",
    l0="medium"
)

prompt = "The quick brown fox jumped over the lazy dog"
inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")
sae_input = gather_attn_out_activations(model, LAYER, inputs)

sae_acts = attn_sae.encode(sae_input.to(torch.float32))
recon = attn_sae.decode(sae_acts)

mse = torch.mean((recon[1:] - sae_input[1:].float())**2)
var = sae_input[1:].float().var()
fvu = mse / var
l0 = (sae_acts[1:] > 0).float().sum(-1).mean()

print(f"L0: {l0:.2f}")
print(f"Fraction of variance unexplained: {mse / var:.2%}")

attn_out/layer_17_width_16k_l0_medium/pa(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

L0: 108.44
Fraction of variance unexplained: 9.99%


## Crosscoders

First, we need a new architecture, to handle stacked encoder matrices (each for a different layer) and all-to-all decoder matrices (which can map from earlier to later layers).

In [ ]:
class JumpReLUMultiLayerSAE(nn.Module):
  def __init__(self, d_in, d_sae, num_layers, affine_skip_connection=False):
    # Note that we initialise these to zeros because we're loading in pre-trained weights.
    # If you want to train your own SAEs then we recommend using blah
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(num_layers, d_in, d_sae))
    self.w_dec = nn.Parameter(torch.zeros(num_layers, d_sae, num_layers, d_in))
    self.threshold = nn.Parameter(torch.zeros(num_layers, d_sae))
    self.b_enc = nn.Parameter(torch.zeros(num_layers, d_sae))
    self.b_dec = nn.Parameter(torch.zeros(num_layers, d_in))
    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(num_layers, d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    pre_acts = einops.einsum(
        input_acts, self.w_enc, "... layer d_in, layer d_in d_sae -> ... layer d_sae"
    ) + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return einops.einsum(
        acts, self.w_dec, "... layer_in d_sae, layer_in d_sae layer_out d_dec -> ... layer_out d_dec"
    ) + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + einops.einsum(
          x, self.affine_skip_connection, "... layer d_in, layer d_in d_dec -> ... layer d_dec"
      )
    return recon

Next, some new logic for loading in our params (which are split over layers):

In [ ]:
NUM_LAYERS = 26

def load_multi_layer_sae(
    category: Literal["clt", "crosscoder"],
    num_layers: int,
    width: str,
    l0: str,
    affine: bool = False,
    device = "cuda",
    half_precision: bool = False,
) -> JumpReLUMultiLayerSAE:

  affine_str = "_affine" if affine else ""

  params_list = []

  if category == "crosscoder":
    # Crosscoder names are e.g. "layer_7_13_17_22_width_262k_l0_medium"
    layers = [int(x * num_layers + 0.5) for x in [0.25, 0.5, 0.65, 0.85]]
    print(f"Loading crosscoder for layers {layers}")
    subcategory = f"layer_{'_'.join(str(x) for x in layers)}_width_{width}_l0_{l0}{affine_str}"
  else:
    assert category == "clt"
    # CLT names are just e.g. "width_262k_l0_medium_affine"
    print("Loading CLT for all layers")
    layers = list(range(num_layers))
    affine_str = "_affine" if affine else ""
    subcategory = f"width_{width}_l0_{l0}{affine_str}"

  for layer_idx in range(len(layers)):
    path_to_params = hf_hub_download(
        repo_id="google/gemma-scope-2-1b-pt",
        filename=f"{category}/{subcategory}/params_layer_{layer_idx}.safetensors",
    )
    params = load_file(path_to_params, device=device)
    params_list.append(params)

  # We stack all params along the leading "layer" dimension
  params = {
      k: torch.stack([params[k] for params in params_list])
      for k in params_list[0].keys()
  }
  d_model, d_sae = params["w_enc"].shape[1:]
  sae = JumpReLUMultiLayerSAE(d_model, d_sae, len(layers), affine)
  sae.load_state_dict(params)
  if half_precision:
    sae = sae.half()
  return sae

And now we can test out our crosscoder, which was trained on layers (7, 13, 17, 22), chosen because these are spaced 25%, 50%, 65% and 85% of the way through the model.

In [ ]:
crosscoder = load_multi_layer_sae(
    category="crosscoder",
    num_layers=NUM_LAYERS,
    width="262k",
    l0="medium"
)

{k: v.shape for k, v in crosscoder.named_parameters()}

Loading crosscoder for layers [7, 13, 17, 22]


crosscoder/layer_7_13_17_22_width_250k_l(…):   0%|          | 0.00/1.51G [00:00<?, ?B/s]

crosscoder/layer_7_13_17_22_width_250k_l(…):   0%|          | 0.00/1.51G [00:00<?, ?B/s]

crosscoder/layer_7_13_17_22_width_250k_l(…):   0%|          | 0.00/1.51G [00:00<?, ?B/s]

crosscoder/layer_7_13_17_22_width_250k_l(…):   0%|          | 0.00/1.51G [00:00<?, ?B/s]

{'w_enc': torch.Size([4, 1152, 65536]),
 'w_dec': torch.Size([4, 65536, 4, 1152]),
 'threshold': torch.Size([4, 65536]),
 'b_enc': torch.Size([4, 65536]),
 'b_dec': torch.Size([4, 1152])}

In [ ]:
crosscoder.cuda()

JumpReLUMultiLayerSAE()

Let's look at its performance too:

In [ ]:
def gather_crosscoder_activations(model, target_layers, inputs):
  act_cache = {}
  handles = []
  for layer in target_layers:
    handle = model.model.layers[layer].register_forward_hook(
        partial(gather_acts_hook, key=f"acts_{layer}", cache=act_cache, use_input=False)
    )
    handles.append(handle)
  try:
    _ = model.forward(inputs)
  finally:
    for handle in handles:
      handle.remove()
  return torch.stack([act_cache[f"acts_{layer}"][0] for layer in target_layers], axis=-2)

In [ ]:
layers = [7, 13, 17, 22]

prompt = "The quick brown fox jumped over the lazy dog"
inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")
sae_input = gather_crosscoder_activations(model, layers, inputs).to(torch.float32)

sae_acts = crosscoder.encode(sae_input)
recon = crosscoder.forward(sae_input)

mse = torch.mean((recon[1:] - sae_input[1:].float())**2)
var = sae_input[1:].float().var()
fvu = mse / var
l0 = (sae_acts[1:] > 0).float().sum((-1, -2)).mean()  # sum over (layer, feature) dims

print(f"L0: {l0:.2f}")
print(f"Fraction of variance unexplained: {mse / var:.2%}")

L0: 75.56
Fraction of variance unexplained: 4.49%


We can also see what its delta loss is, when we intervene on a single layer:

In [ ]:
# Forward pass to get logits & hidden activations
model_output_clean = model.forward(inputs_physics, output_hidden_states=True)
logits_clean = model_output_clean.logits[0]  # (seq, d_vocab)

# Extract all crosscoder input activations
input_acts = torch.stack([
    model_output_clean.hidden_states[layer + 1][0]
    for layer in layers
], axis=1)  # (seq, layers, d_model)

# Get the SAE reconstruction
recon = crosscoder.forward(input_acts.to(torch.float32))

def intervene_with_crosscoder(mod, inputs, outputs, layer):
  outputs[0][layers.index(layer), 1:] = recon[1:, layers.index(layer)]
  return outputs

# Add hooks to intervene at every layer we plan to get the loss from
handles = []
for layer in layers:
  handle = model.model.layers[layer].register_forward_hook(
      partial(intervene_with_crosscoder, layer=layer)
  )
  handles.append(handle)
try:
  model_output = model.forward(
      einops.repeat(inputs_physics, "1 seq -> 4 seq")
  )
finally:
  for handle in handles:
    handle.remove()

# Get logits from this corrupted forward pass
logits_sae = model_output.logits[0]

loss_clean = cross_entropy_loss(logits_clean, inputs_physics[0])
print(f"Loss (clean): {loss_clean.mean():.4f}")

for label, logits in zip(layers, model_output.logits):
  loss_sae = cross_entropy_loss(logits, inputs_physics[0])
  delta_loss = loss_sae - loss_clean
  print(f"Delta loss at layer {label:02}: {delta_loss.mean():.4f}")

Loss (clean): 1.3977
Delta loss at layer 07: 0.1484
Delta loss at layer 13: 0.7947
Delta loss at layer 17: 0.1789
Delta loss at layer 22: 0.4761


For a given L0 (in this case 50), we expect crosscoders to have a higher delta loss per layer than the corresponding SAE, because their sparsity budget has to be allocated across all layers.

## CLTs

Finally we'll look at CLTs - we've done most of the work for them already, since their architecture is similar to crosscoders and their activation sites are equivalent to transcoders (except concatenated).

In [ ]:
# Code to delete previous models, freeing up space:
try:
  del crosscoder, attn_sae, mlp_sae, transcoder, sae
except NameError:
  print("Already deleted.")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
clt = load_multi_layer_sae(
    category="clt",
    num_layers=NUM_LAYERS,
    width="262k",
    l0="big",
    affine=True,
    half_precision=True,
)

{k: v.shape for k, v in clt.named_parameters()}

Loading CLT for all layers


clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

...

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

clt/width_262080_affine_l0_big/params_la(…):   0%|          | 0.00/1.26G [00:00<?, ?B/s]

{'w_enc': torch.Size([26, 1152, 10080]),
 'w_dec': torch.Size([26, 10080, 26, 1152]),
 'threshold': torch.Size([26, 10080]),
 'b_enc': torch.Size([26, 10080]),
 'b_dec': torch.Size([26, 1152]),
 'affine_skip_connection': torch.Size([26, 1152, 1152])}

In [ ]:
clt.cuda()

JumpReLUMultiLayerSAE()

In [ ]:
def gather_clt_activations(model, num_layers, inputs):
  act_cache = {}
  handles = []
  for layer in range(num_layers):
    handle_input = model.model.layers[layer].pre_feedforward_layernorm.register_forward_hook(
        partial(gather_acts_hook, cache=act_cache, key=f"input_{layer}", use_input=False)
    )
    handle_target = model.model.layers[layer].post_feedforward_layernorm.register_forward_hook(
        partial(gather_acts_hook, cache=act_cache, key=f"target_{layer}", use_input=False)
    )
    handles.extend([handle_input, handle_target])
  try:
    _ = model.forward(inputs)
  finally:
    for handle in handles:
      handle.remove()

  return (
      torch.stack([act_cache[f"input_{layer}"] for layer in range(num_layers)], axis=-2),
      torch.stack([act_cache[f"target_{layer}"] for layer in range(num_layers)], axis=-2),
  )

In [ ]:
sae_input, sae_target = gather_clt_activations(model, NUM_LAYERS, inputs_physics)
sae_input = sae_input.half()
sae_target = sae_target.half()

sae_acts = clt.encode(sae_input)
recon = clt.forward(sae_input)

mse = torch.mean((recon[1:] - sae_target[1:].float())**2)
var = sae_target[1:].float().var()
fvu = mse / var
l0 = (sae_acts[1:] > 0).float().sum((-1, -2)).mean()  # sum over (layer, feature) dims

print(f"L0: {l0:.2f}")
print(f"Fraction of variance unexplained: {mse / var:.2%}")

L0: 258.44
Fraction of variance unexplained: 30.72%


This looks like quite a high fraction of variance unexplained, but digging deeper we find this is because the FVU is much higher for later layers than it is for early layers (and late-layer MLP outputs have much higher norm). To test whether our CLT is effectively capturing the important part of the late-layer MLP outputs, we can look at the delta loss from intervening at each layer:

In [ ]:
# Forward pass to get logits & hidden activations
model_output_clean = model.forward(inputs_physics, output_hidden_states=True)
logits_clean = model_output_clean.logits[0]  # (seq, d_vocab)

# Extract all crosscoder input activations
sae_input, sae_target = gather_clt_activations(model, NUM_LAYERS, inputs_physics)

# Get the SAE reconstruction
recon = clt.forward(sae_input.half())

def intervene_with_clt(mod, inputs, outputs, layer):
  outputs[layer, 1:] = recon[1:, layer]
  return outputs

# Add hooks to intervene at every layer we plan to get the loss from
handles = []
for layer in range(NUM_LAYERS):
  handle = model.model.layers[layer].post_feedforward_layernorm.register_forward_hook(
      partial(intervene_with_clt, layer=layer)
  )
  handles.append(handle)
try:
  model_output = model.forward(
      einops.repeat(inputs_physics, f"1 seq -> {NUM_LAYERS} seq")
  )
finally:
  for handle in handles:
    handle.remove()

# Get logits from this corrupted forward pass
logits_sae = model_output.logits[0]

loss_clean = cross_entropy_loss(logits_clean, inputs_physics[0])
print(f"Loss (clean): {loss_clean.mean():.4f}")

for layer, logits in enumerate(model_output.logits):
  loss_sae = cross_entropy_loss(logits, inputs_physics[0])
  delta_loss = loss_sae - loss_clean
  print(f"Delta loss at layer {layer:02}: {delta_loss.mean():.4f}")

Loss (clean): 1.3977
Delta loss at layer 00: 0.0308
Delta loss at layer 01: -0.0392
Delta loss at layer 02: -0.0073
Delta loss at layer 03: 0.0002
Delta loss at layer 04: -0.0247
Delta loss at layer 05: -0.0215
Delta loss at layer 06: 0.0390
Delta loss at layer 07: 0.0046
Delta loss at layer 08: -0.0482
Delta loss at layer 09: 0.0373
Delta loss at layer 10: 0.0553
Delta loss at layer 11: 0.0636
Delta loss at layer 12: 0.0475
Delta loss at layer 13: 0.0136
Delta loss at layer 14: 0.0041
Delta loss at layer 15: 0.0191
Delta loss at layer 16: 0.1631
Delta loss at layer 17: 0.1398
Delta loss at layer 18: 0.1480
Delta loss at layer 19: 0.0974
Delta loss at layer 20: -0.0268
Delta loss at layer 21: 0.0440
Delta loss at layer 22: -0.0008
Delta loss at layer 23: 0.0333
Delta loss at layer 24: 0.0232
Delta loss at layer 25: 0.0482


## IT SAEs & displaying top activations

Here, you can load in example activations data to see what kinds of prompts maximally cause a particular feature to fire.

When we finish adding full Neuronpedia support, we'll add to this setion!

First, here's a function to load data from HuggingFace:

In [ ]:
def load_example_data(
    model_size: str = "1b",
    category: str = "resid_post",
    layer: int = 0,
    width: int = "16k",
    l0: str = "medium",
    affine: bool = False,
    instruction_tuned: bool = True,
) -> dict[str, np.ndarray]:
  affine_str = "_affine" if affine else ""
  repo_id=f"google/gemma-scope-2-{model_size}-{'it' if instruction_tuned else 'pt'}"
  filename=f"{category}/layer_{layer}_width_{width}_l0_{l0}{affine_str}/examples.safetensors"
  print(repo_id)
  print(filename)
  path_to_data = hf_hub_download(
      repo_id=f"google/gemma-scope-2-{model_size}-{'it' if instruction_tuned else 'pt'}",
      filename=f"{category}/layer_{layer}_width_{width}_l0_{l0}{affine_str}/examples.safetensors",
  )
  return load_file(path_to_data)

Next, some helper functions to help us visualize the data:

In [ ]:
def to_str_tokens(tokens: list[int]) -> list[str]:
  str_tokens = tokenizer.convert_ids_to_tokens(tokens)
  for i, t in enumerate(str_tokens):
    if t.startswith("▁"):
      str_tokens[i] = " " + t[1:]
  return str_tokens


def span(str_tok, act, logit, max_logit):
  # Activtion determines background colour
  bg_color = f"rgba(0,255,0,{act:.3f})"
  # Logit effect determines underline color: blue (+ve), red (-ve)
  logit_normed = logit / max_logit if max_logit > 1e-9 else 0.0
  logit_normed = max(-1.0, min(1.0, logit_normed))
  u_color = (
      f"rgba(0,0,255,{logit_normed:.3f})"
      if logit_normed >= 0.0
      else f"rgba(0,0,255,{-logit_normed:.3f})"
  )
  # Use thick bottom border for underline
  style = f"background-color: {bg_color}; border-bottom: 3px solid {u_color};"
  return f'<span style="{style}">{str_tok}</span>'


def join_str_tok_list(str_toks, acts, logits, max_logit):
  str_toks = [x.replace("\n", "⏎") for x in str_toks]
  logits = [0.0] + logits.tolist()[:-1]  # logit effect is for the *next* token
  return "".join([span(*args, max_logit) for args in zip(str_toks, acts, logits, strict=True)])

escape_html = lambda x: x.replace("<", "&lt;").replace(">", "&gt;")


def inspect_feature(
    example_data: dict[str, np.ndarray],
    feature_id: int,
    buf: tuple[int, int] = (25, 25),
    max_examples: int = 10,
    reuse_same_sequences: bool = True,
    max_logit_effect: float | None = None,
) -> None:
  """Visualizes the top-activating sequences for a particular feature."""
  tokens = example_data["tokens"]
  activations = example_data["activations"]
  positions = example_data["positions"]
  seq_ids = example_data["seq_ids"]
  feature_frequencies = example_data["feature_frequencies"]
  logit_effects = example_data["logit_effects"]

  # Get activations, cropped past the point where it's not actually active (we
  # just padded the array out to the same length as the other features)
  activations = activations[feature_id]
  n_acts = (activations > 0).sum().item()
  if n_acts == 0:
    print(f"No activations for feature {feature_id}")
    return
  activations = activations[:n_acts]
  seq_ids = seq_ids[feature_id][:n_acts]
  positions = positions[feature_id][:n_acts]
  logit_effects = logit_effects[feature_id][:n_acts]
  if max_logit_effect is None:
    max_logit_effect = np.abs(logit_effects).max().item()

  # Print out frequency in 2 different ways (cached value & dataframe counts)
  print(f"Inspecting feature {feature_id}")
  print(f"Frequency: {feature_frequencies[feature_id]:.2e}")
  if "top_tokens" in example_data:
    str_tokens = to_str_tokens(example_data["top_tokens"][feature_id])
    print(f"Top tokens: {str_tokens}")

  # Make a list of formatted sequences for each of our top activations
  top_activations = []
  formatted_sequences = []
  position_tuples = []
  max_act = max(activations[0], 1e-12)
  while len(formatted_sequences) < max_examples:
    # Finish if we don't have any more nonzero examples we can take
    if not seq_ids.shape[0]:
      break

    # Pick the max-activation sequence not yet chosen
    idx = np.argmax(activations).item()
    seq_id = seq_ids[idx].item()
    position = positions[idx].item()
    activation = activations[idx].item()
    position_tuples.append(f"{seq_id},{position}")
    top_activations.append(activation)

    # Get the string tokens, maybe adjusting the buffer if this token is too
    # close to the start or end of the sequence
    true_buf = (
        min(buf[0], position),
        min(buf[1], tokens.shape[1] - 1 - position),
    )
    str_toks = to_str_tokens(
        tokens[seq_id, position - true_buf[0] : position + true_buf[1] + 1]
    )

    # Initialize buffers for activations and logit effects
    seq_len_window = true_buf[1] + true_buf[0] + 1
    acts = np.zeros((seq_len_window,))
    logits = np.zeros((seq_len_window,))
    str_toks = list(map(escape_html, str_toks))

    # Get the tokens & activations in a padded region around that sequence
    seq_id_mask = seq_ids == seq_id
    pos_diff = positions - position
    position_mask = (-pos_diff < true_buf[0]) & (pos_diff < true_buf[1])
    full_mask = seq_id_mask & position_mask

    # Apply mask to both activations and logit effects
    for pos, act, logit in zip(positions[full_mask], activations[full_mask], logit_effects[full_mask]):
      offset = pos - position + true_buf[0]
      acts[offset] = act / max_act
      logits[offset] = logit
    formatted_sequences.append(join_str_tok_list(str_toks, acts, logits, max_logit_effect))

    # Filter out all other activations with the same sequence
    filter_mask = ~(full_mask if reuse_same_sequences else seq_id_mask)
    activations = activations[filter_mask]
    seq_ids = seq_ids[filter_mask]
    positions = positions[filter_mask]
    logit_effects = logit_effects[filter_mask]

  output_df = pd.DataFrame({
      "position": position_tuples,
      "activation": top_activations,
      "tokens": formatted_sequences,
  }).reset_index(drop=True)
  display(
      output_df.style.set_properties(
          subset=["tokens"], **{"font-family": "Helvetica"}
      )
  )

Now we can inspect our thermodynamics feature from earlier:

<!-- Now we can inspect a feature which seems to fire on mentions of **conspiracy theories**, in Gemma V3 27b PT: 50705 -->

In [ ]:
example_data = load_example_data(
    model_size="1b",
    category="resid_post",
    width="65k",
    l0="medium",
    layer=22,
    instruction_tuned=False,
)

inspect_feature(example_data, feature_id=6524)

google/gemma-scope-2-1b-pt
resid_post/layer_22_width_65k_l0_medium/examples.safetensors


resid_post/layer_22_width_65k_l0_medium/(…):   0%|          | 0.00/1.72G [00:00<?, ?B/s]

/tmp/ipython-input-164843109.py:61: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



Inspecting feature 6524
Frequency: 2.64e-04
Top tokens: [' entropy', ' Entropy', 'Entropy', ' entrop', ' enthalpy', ' calorimetric', ' Gibb', '焓', ' Gibbs', ' enthal']


We can also see how this feature looks in the instruction-tuned variant. Does it seem to fire on the same kinds of topics? (Generally we expect the answer is yes given the IT versions were finetuned from PT versions, unless e.g. the feature is very chat-specific).

In [ ]:
example_data = load_example_data(
    model_size="1b",
    category="resid_post",
    width="65k",
    l0="medium",
    layer=22,
    instruction_tuned=True,
)

inspect_feature(example_data, feature_id=6524)

google/gemma-scope-2-1b-it
resid_post/layer_22_width_65k_l0_medium/examples.safetensors


resid_post/layer_22_width_65k_l0_medium/(…):   0%|          | 0.00/1.81G [00:00<?, ?B/s]

Inspecting feature 6524
Frequency: 8.86e-04
Top tokens: [' entropy', ' Entropy', '熵', 'entropy', 'Entropy', ' disorder', ' entrop', ' irrevers', '樀', '熱']


/tmp/ipython-input-164843109.py:61: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [29]:
print("Hello from Scribe test!")

Hello from Scribe test!
